In [ ]:
# QA Assistant code

from google.colab import drive
drive.mount('/content/drive')

!pip install openai==0.27.2 requests pandas
!pip install jira
!pip install python-dotenv

from dotenv import load_dotenv
load_dotenv("/content/drive/MyDrive/Colab_Notebooks/projectQaAssistant/.env")

import os
import base64
import requests
import openai
from jira import JIRA
import re

# Constants and environment
TESTOMAT_BASE_URL = "https://app.testomat.io/api"
PROJECT_ID = "qa_assistant"
SUITE_ID = "49f4dca5"
JIRA_BASE_URL = os.getenv("JIRA_BASE_URL")
CONFLUENCE_BASE_URL = os.getenv("CONFLUENCE_BASE_URL")

# Load API keys
openai.api_key = os.getenv("OPENAI_API_KEY")
ATLASSIAN_EMAIL = os.getenv("ATLASSIAN_EMAIL")
ATLASSIAN_API_KEY = os.getenv("ATLASSIAN_API_KEY")
TESTOMAT_API_KEY = os.getenv("TESTOMAT_API_KEY")

def get_basic_auth_header(email, token):
    auth_string = f"{email}:{token}"
    return base64.b64encode(auth_string.encode()).decode()

def check_openai_connection():
    try:
        response = openai.ChatCompletion.create(
            model="gpt-4",
            messages=[{"role": "system", "content": "Ping"}]
        )
        print("✅ OpenAI API connected")
    except Exception as e:
        print(f"❌ OpenAI connection error: {e}")

def check_atlassian_connection():
    try:
        headers = {
            "Authorization": f"Basic {get_basic_auth_header(ATLASSIAN_EMAIL, ATLASSIAN_API_KEY)}",
            "Accept": "application/json"
        }
        response = requests.get(f"{JIRA_BASE_URL}/rest/api/3/myself", headers=headers)
        response.raise_for_status()
        print("✅ Atlassian API connected")
    except Exception as e:
        print(f"❌ Atlassian connection error: {e}")

def get_jira_ticket_info(ticket_id):
    try:
        jira = JIRA(basic_auth=(ATLASSIAN_EMAIL, ATLASSIAN_API_KEY), options={'server': JIRA_BASE_URL})
        issue = jira.issue(ticket_id)
        return issue.fields.summary, issue.fields.description or "No description provided"
    except Exception as e:
        print(f"❌ JIRA issue fetch failed for {ticket_id}: {e}")
        return None, None

def generate_openai_content(prompt, role_desc):
    try:
        response = openai.ChatCompletion.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": role_desc},
                {"role": "user", "content": prompt}
            ]
        )
        return response['choices'][0]['message']['content'].strip()
    except Exception as e:
        print(f"❌ OpenAI generation failed: {e}")
        return None

def create_confluence_page(title, content, space_key="qaAssistan"):
    try:
        headers = {
            "Authorization": f"Basic {get_basic_auth_header(ATLASSIAN_EMAIL, ATLASSIAN_API_KEY)}",
            "Content-Type": "application/json"
        }
        data = {
            "title": title,
            "space": {"key": space_key},
            "type": "page",
            "body": {"storage": {"value": content, "representation": "storage"}}
        }
        response = requests.post(f"{CONFLUENCE_BASE_URL}/rest/api/content", json=data, headers=headers)
        response.raise_for_status()
        print(f"✅ Confluence page created: {CONFLUENCE_BASE_URL}/pages/{response.json()['id']}")
    except Exception as e:
        print(f"❌ Confluence page creation failed: {e}")

def get_testomat_jwt_token():
    try:
        response = requests.post(f"{TESTOMAT_BASE_URL}/login", data={"api_token": TESTOMAT_API_KEY})
        response.raise_for_status()
        return response.json().get("jwt")
    except Exception as e:
        print(f"❌ Testomat token error: {e}")
        return None

def create_test_cases_in_testomat(test_cases, jwt_token):
    headers = {"Authorization": jwt_token, "Content-Type": "application/json"}

    # Розділяємо блоки за початком кожного тест-кейсу
    test_case_blocks = re.split(r'\n(?=###\s+Перевірити, що)', test_cases.strip())
    created_test_ids = []

    for block in test_case_blocks:
        block = block.strip()
        if not block:
            continue

        # Отримуємо назву з першого заголовка
        title_match = re.search(r'^###\s+Перевірити, що[^\n]+', block)
        title = title_match.group(0).replace("###", "").strip() if title_match else "Без назви"

        data = {
            "data": {
                "type": "tests",
                "attributes": {
                    "title": title,
                    "description": block,
                    "suite-id": SUITE_ID,
                    "priority": 0
                }
            }
        }

        try:
            response = requests.post(f"{TESTOMAT_BASE_URL}/{PROJECT_ID}/tests", json=data, headers=headers)
            response.raise_for_status()
            test_id = response.json()["data"]["id"]
            created_test_ids.append(test_id)
            print(f"✅ Створено тест: {title} (ID: {test_id})")
        except Exception as e:
            print(f"❌ Помилка при створенні тесту '{title}': {e}")

    if created_test_ids:
        print(f"\n📌 Успішно створено {len(created_test_ids)} тестів.")
    else:
        print("⚠️ Жоден тест не було створено.")

def main():
    check_openai_connection()
    check_atlassian_connection()

    ticket_id = input("Enter Jira ticket ID (e.g., SCRUM-5): ").strip()
    summary, description = get_jira_ticket_info(ticket_id)

    if not summary or not description:
        print("❌ Missing summary or description.")
        return

    doc_prompt = f"""
<reasoning>
- Simple Change: no
- Reasoning: yes
    - Identify: “інтерпретує вхідні дані та формує структуру” + “виводить секції документації”
    - Conclusion: yes
    - Ordering: after
- Structure: yes
- Examples: yes (as reference, not enforcement)
    - Representative: 4
- Complexity: 5
    - Task: 5
    - Necessity: high
- Specificity: 5
- Prioritization: ["Freedom of Structure", "Reasoning", "Audience Awareness"]
- Conclusion: Allow GPT to choose which additional sections to include; reduce strictness of examples; preserve HTML output format; instruct the model to act as a highly competent QA with cross-functional awareness.
</reasoning>

Generate structured documentation in Ukrainian for insertion into Confluence using Jira ticket data. The documentation must be in HTML format compatible with Confluence ("representation": "storage").

Your audience includes QA engineers, product managers, support and marketing teams. Focus on describing the **logic of functionality** and **important technical implementation details**, but also provide clarity for non-technical readers when relevant.

You are allowed to add **additional sections** or highlight important things you believe are **relevant** or **often overlooked** in specifications — think and write like a highly competent senior QA.

If the ticket description is incomplete, reasonably infer missing data based on the title and context. Do not hallucinate — stay aligned with the technical domain.

# Guidelines

- Language: Ukrainian
- Tone: Professional, clear, informative.
- Format: Use HTML tags compatible with Confluence:
    - `<h1>`: document title
    - `<h2>`: major sections
    - `<p>`: paragraphs
    - `<ul><li>`: bullet lists
    - `<pre><code>`: for API examples or code
    - `<strong>`: highlight terms
- Structure: Include standard sections if applicable (logic, purpose, scenarios, API examples, glossary). Add new sections if relevant.
- Fallback: If data is missing — enrich content with domain-specific knowledge.

# Output Format

Return fully formatted HTML. No markdown, no JSON. Must be valid for insertion into Confluence API.

# Examples

(These are only examples. Do not follow rigidly — adapt based on input.)

<h1>Оновлення профілю користувача</h1>

<h2>Опис логіки</h2>
<p>Користувач може оновити свій email, ім’я або пароль через інтерфейс профілю.</p>

<h2>Очікувана поведінка</h2>
<ul>
  <li>Зміни зберігаються після натискання кнопки "Зберегти"</li>
</ul>

<h2>API</h2>
<pre><code>PUT /api/user/update
Body:
{{ "email": "new@example.com" }}</code></pre>

<h2>Глосарій</h2>
<ul>
  <li><strong>JWT</strong> — токен авторизації</li>
</ul>

# Notes

- Think critically about what’s useful to include.
- You are free to restructure, omit, or expand beyond the example.
- Always maintain valid HTML output.
---

📌 Назва задачі: {summary}
📝 Опис задачі: {description}
"""

    confluence_content = generate_openai_content(doc_prompt, "Ти Senior QA Engineer.")
    if confluence_content:
        create_confluence_page(summary, confluence_content)

    test_prompt = test_prompt = f"""
<reasoning>
- Simple Change: no
- Reasoning: yes
    - Identify: модель має аналізувати логіку задачі, виявляти параметри, умови, переходи
    - Conclusion: yes
    - Ordering: reasoning → classification → structured cases
- Structure: yes
- Examples: yes
    - Representative: 5
- Complexity: 5
- Specificity: 5
- Prioritization: ["Test Design Techniques", "Readable Case Titles", "Markdown Output"]
- Conclusion: Use formal test design methods (BVA, EP, decision table, state, etc.) to generate structured, markdown-formatted test cases with rule-based naming in Ukrainian.
</reasoning>

You are a Senior QA Engineer. Based on the provided Jira summary and description, generate a set of test cases **using formal test design techniques** (such as Boundary Value Analysis, Equivalence Partitioning, State Transition Testing, Decision Table Testing, etc.).

Each test case must:
- Be written in **Ukrainian**
- Have a **clear, rule-based name** starting with “Перевірити, що...”
- Be formatted using **Markdown**, fully compatible with Testomat.io
- Include clear sections:
  - Flag (optional)
  - Preconditions (bullet or numbered)
  - Steps (in a Markdown table)
- Reflect actual logic and possible edge cases
- Supplement missing logic using domain knowledge, if needed

# Output Structure per Test Case (repeat for 3–5 cases)

Each test case must cover **one distinct logical scenario** and be a **complete unit** with its own **title**, **preconditions**, and **steps**.

### Verify that [describe what should be verified]

### Flag
[OPTIONAL: Feature flag name] - ON

### Preconditions

1. [List the system/user preconditions before the test starts]
2. [Any setup required for the feature to work]

### Steps

Use a Markdown table to show actions and their corresponding expected results:

|#| Step | Expected result |
|---|---|---|
|1.| [User action] | [System reaction or visible output] |
|2.| [Next action] | [Next expected outcome] |

# Guidelines

- Use test design techniques:
    - **Boundary Value Analysis (BVA)** for numeric or range-based inputs
    - **Equivalence Partitioning (EP)** for grouping similar values
    - **Decision Tables** when outcomes depend on rules/conditions
    - **State Transition** for flows or navigation
    - **Pairwise** if many parameters interact
- Add edge cases when appropriate
- Highlight UI elements or messages with `**bold**` where relevant
- Separate test cases using two line breaks (`\n\n`)
- Do not use JSON or plain text lists

---

📌 Назва задачі: {summary}
📝 Опис задачі: {description}
"""

    test_cases = generate_openai_content(test_prompt, "Ти досвідчений QA Engineer.")
    if test_cases:
        jwt = get_testomat_jwt_token()
        if jwt:
            create_test_cases_in_testomat(test_cases, jwt)

if __name__ == "__main__":
    main()